# 卡方檢定特徵篩選工具

## 工具目的

從類別型特徵中篩選與目標變數（如 `Survived`）有**顯著統計關聯**的特徵，作為特徵選擇的依據。

## 使用場景

在特徵選擇階段，面對多個類別型變數時，需要客觀判斷哪些變數值得納入模型。

## 核心價值

- 用**統計檢定**取代直覺判斷
- 搭配 **Bonferroni 校正**控制多重比較的偽陽性（Family-Wise Error Rate）
- 提供**效應量**（Cramér's V）避免「統計顯著但實際影響微小」的陷阱

## 卡方獨立性檢定原理

卡方檢定比較列聯表中的**觀察頻率 (O)** 與假設兩變數獨立時的**期望頻率 (E)**：

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

- 若 $\chi^2$ 很大 → 觀察值與期望值偏離大 → 兩變數**不獨立**（有關聯）
- 若 $\chi^2$ 接近 0 → 觀察值符合期望 → 兩變數**獨立**（無關聯）

## Cramér's V（效應量指標）

$$V = \sqrt{\frac{\chi^2}{n \times (\min(r, c) - 1)}}$$

- 範圍：0（完全無關聯）~ 1（完全關聯）
- 判讀標準：
  - V < 0.1：小效應（效果微弱）
  - 0.1 ≤ V < 0.3：中效應
  - V ≥ 0.3：大效應（實質影響）

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns

## 載入資料

使用 Kaggle Titanic 訓練集，包含乘客基本資訊與存活結果。

In [ ]:
df = pd.read_csv('./train.csv', encoding='utf-8-sig')
df.info()
df.head()

## 改進版卡方檢定函數

相較於基礎版（僅回傳顯著特徵名稱），改進版增加：

| 改進項目 | 說明 |
|---------|------|
| 完整統計量 | 回傳 χ²、p-value、自由度、校正後 α |
| 效應量 | 計算 Cramér's V，判斷實際影響大小 |
| 假設檢查 | 驗證期望頻率 ≥ 5 的前提假設是否滿足 |
| 缺失值處理 | 逐特徵移除缺失值，不影響其他特徵的計算 |
| 結構化輸出 | 回傳 DataFrame，方便後續分析與排序 |

In [ ]:
def chi_square_feature_selection(df, features, target, alpha=0.05):
    """
    對每個特徵執行卡方獨立性檢定，並應用 Bonferroni 校正。

    Parameters
    ----------
    df : pd.DataFrame
        輸入資料集
    features : list of str
        要檢定的特徵欄位名稱
    target : str
        目標變數欄位名稱
    alpha : float
        原始顯著水準（預設 0.05）

    Returns
    -------
    pd.DataFrame
        包含 chi2, p-value, corrected alpha, Cramér's V,
        assumption check 等欄位，依 p-value 排序
    """
    results = []
    n_tests = len(features)
    corrected_alpha = alpha / n_tests  # Bonferroni correction

    for feature in features:
        # 移除該特徵的缺失值（不影響其他特徵）
        subset = df[[feature, target]].dropna()

        # 建立列聯表 (contingency table)
        contingency = pd.crosstab(subset[feature], subset[target])

        # 執行卡方檢定
        chi2, p, dof, expected = chi2_contingency(contingency)

        # 假設檢查：期望頻率 >= 5
        min_expected = expected.min()
        pct_below_5 = (expected < 5).sum() / expected.size * 100
        assumption_met = (
            "Yes"
            if min_expected >= 5
            else f"No (min={min_expected:.1f}, {pct_below_5:.0f}% cells < 5)"
        )

        # 計算 Cramér's V (effect size)
        n = len(subset)
        k = min(contingency.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * k)) if k > 0 else 0

        # 判斷顯著性（使用 Bonferroni 校正後的 alpha）
        significant = p < corrected_alpha

        results.append({
            'Feature': feature,
            'Chi2': round(chi2, 4),
            'p-value': p,
            'Corrected \u03b1': round(corrected_alpha, 6),
            'Significant': '是' if significant else '否',
            "Cram\u00e9r's V": round(cramers_v, 4),
            'Effect Size': (
                'Small' if cramers_v < 0.1
                else ('Medium' if cramers_v < 0.3 else 'Large')
            ),
            'DoF': dof,
            'Min Expected': round(min_expected, 2),
            'Assumption Met': assumption_met,
        })

    return pd.DataFrame(results).sort_values('p-value')

## 執行特徵篩選

選擇常見的類別型 / 序數型特徵進行卡方檢定：

- `Pclass`：船票等級（1/2/3），序數型
- `Sex`：性別，二元類別
- `SibSp`：船上兄弟姊妹/配偶數
- `Parch`：船上父母/子女數
- `Embarked`：登船港口（C/Q/S）

In [ ]:
# 選擇要檢定的特徵（排除 target 和連續變數）
test_features = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked']

results = chi_square_feature_selection(df, test_features, 'Survived')
results

## 結果解讀

### 如何閱讀結果表

| 欄位 | 意義 |
|------|------|
| **Chi2** | 卡方統計量，越大代表觀察值與期望值偏離越大 |
| **p-value** | 在虛無假設（兩變數獨立）下觀察到此結果的機率 |
| **Corrected α** | Bonferroni 校正後的顯著水準（= 0.05 / 檢定次數） |
| **Significant** | p-value < Corrected α 時為「是」 |
| **Cramér's V** | 效應量，衡量關聯的實際強度 |
| **Assumption Met** | 卡方檢定前提假設（期望頻率 ≥ 5）是否滿足 |

### Cramér's V 判讀標準

| V 值範圍 | 效應大小 | 實務意義 |
|----------|---------|----------|
| < 0.1 | Small | 統計顯著但實際影響微弱，建模價值低 |
| 0.1 ~ 0.3 | Medium | 有實質關聯，值得納入模型 |
| ≥ 0.3 | Large | 強關聯，重要特徵 |

## 視覺化：列聯表熱力圖

針對顯著特徵，繪製正規化後的列聯表熱力圖，直觀呈現各類別的存活率差異。

In [ ]:
significant_features = results[results['Significant'] == '是']['Feature'].tolist()

n = len(significant_features)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, feat in zip(axes, significant_features):
        ct = pd.crosstab(df[feat], df['Survived'], normalize='index')
        ct.columns = ['死亡', '存活']
        sns.heatmap(ct, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax)
        v = results[results['Feature'] == feat]["Cram\u00e9r's V"].values[0]
        ax.set_title(f"{feat}\nCram\u00e9r's V = {v:.3f}")

    plt.tight_layout()
    plt.show()
else:
    print('沒有顯著特徵可視覺化')

## 延伸：對所有欄位進行篩選

將所有欄位（排除 `Survived`）都納入檢定，觀察哪些欄位即使類別數很多（如 `Name`、`Ticket`），
在統計上是否仍然顯著，以及假設條件是否滿足。

In [ ]:
all_features = [c for c in df.columns if c != 'Survived']

all_results = chi_square_feature_selection(df, all_features, 'Survived')
all_results

## 工具總結與注意事項

### 卡方檢定的前提假設

- 所有**期望頻率 ≥ 5**（非觀察頻率）
- 當假設不滿足時的替代方案：
  - **2×2 表格**：改用 Fisher's exact test（`scipy.stats.fisher_exact`）
  - **較大表格**：合併低頻類別，直到期望頻率滿足條件

### Bonferroni 校正的限制

- Bonferroni 校正**偏保守**，在檢定數量多時容易產生偽陰性
- 替代方案：**Benjamini-Hochberg FDR**（控制 False Discovery Rate），在探索性分析中更常用

### 效應量的重要性

- 大樣本下，即使微小差異也可能「統計顯著」
- Cramér's V 提供**實際影響大小**的判斷依據
- 建議同時考慮 p-value 和 Cramér's V 來決定是否納入特徵

### 適用範圍

- 此工具僅處理**類別 vs. 類別**的關聯
- 連續變數 vs. 類別：需改用 **t 檢定**（兩組）或 **ANOVA**（多組）
- 連續變數 vs. 連續：需改用 **Pearson/Spearman 相關係數**
- 完整統計檢定對照表請參見 `statistic_test_table`